# Sistema de Recomendação de Cursos Online
### Requalificação Profissional — Filtragem Colaborativa e Baseada em Conteúdo

**Projeto Aplicado III — Universidade Presbiteriana Mackenzie**

Bruno Cerqueira Gianotti | Reginaldo Rogério de Campos | Samya Camily de Santana Silva

---

Este notebook reúne, em ordem, todo o pipeline técnico do projeto:

1. Análise Exploratória de Dados (EDA)
2. Limpeza e Pré-processamento
3. Modelo Content-Based (TF-IDF + similaridade de cosseno)
4. Modelo Colaborativo (KNN item-based)
5. Avaliação comparativa (Precision@K, Recall@K)
6. Modelo Híbrido (combinação ponderada dos dois modelos)

**Base de dados:** [Course Reviews on Coursera](https://www.kaggle.com/datasets/imuhammad/course-reviews-on-coursera) (Kaggle)


## 0. Configuração do ambiente

In [ ]:
# Bibliotecas usadas no projeto (o Google Colab ja vem com quase tudo
# pre-instalado; essa linha garante que nada fique faltando).
!pip install -q pandas numpy scikit-learn scipy kagglehub


## 0.1 Carregar os dados

Este notebook tenta **3 formas** de obter os dados, nesta ordem:

1. **Download automático do Kaggle** (via `kagglehub`) — não precisa fazer upload manual,
   mas exige uma credencial de API do Kaggle configurada (ver como obter abaixo).
2. **Upload manual** (se a opção 1 falhar) — sobe os dois CSVs pela interface do Colab.
   💡 **Dica:** o `Coursera_reviews.csv` tem ~260 MB, o que deixa o upload bem lento.
   Comprima-o em **gzip** antes de subir (gera um `Coursera_reviews.csv.gz` de ~33 MB) —
   o notebook aceita o arquivo comprimido do mesmo jeito, sem precisar descompactar.
   No Windows, o 7-Zip faz isso (`Add to archive` → formato `gzip`); no Mac/Linux, o
   comando `gzip -k Coursera_reviews.csv` no terminal.
3. **Arquivos locais** — se você já tiver os CSVs (ou `.csv.gz`) na mesma pasta do notebook.

### Como obter a credencial do Kaggle (só precisa fazer uma vez)

1. Acesse https://www.kaggle.com/settings → seção **API** → botão **Create New Token**.
2. Isso baixa um arquivo `kaggle.json` pro seu computador.
3. No Colab, vá no ícone de **chave 🔑** (Secrets) na barra lateral esquerda e crie dois
   secrets: `KAGGLE_USERNAME` e `KAGGLE_KEY`, com os valores que estão dentro do `kaggle.json`
   (abra o arquivo num editor de texto pra ver — são só duas linhas).
4. Ative o acesso do notebook a esses secrets (toggle "Notebook access").

Se você pular essa configuração, sem problema — o notebook detecta e pede upload manual
automaticamente.


In [ ]:
import os

def localizar_csvs(pasta):
    """Procura recursivamente os arquivos necessarios dentro de uma pasta
    (o kagglehub baixa o dataset numa estrutura de subpastas que pode variar).
    Aceita tanto .csv quanto .csv.gz (arquivo comprimido -- o pandas le os
    dois formatos exatamente do mesmo jeito, sem precisar descompactar)."""
    alvos = {
        "Coursera_courses.csv": ("Coursera_courses.csv", "Coursera_courses.csv.gz"),
        "Coursera_reviews.csv": ("Coursera_reviews.csv", "Coursera_reviews.csv.gz"),
    }
    encontrados = {}
    for raiz, _, arquivos in os.walk(pasta):
        for arq in arquivos:
            for chave, nomes_aceitos in alvos.items():
                if arq in nomes_aceitos and chave not in encontrados:
                    encontrados[chave] = os.path.join(raiz, arq)
    return encontrados

caminhos = {}

# -------------------------------------------------------------
# Opcao 1: baixar direto do Kaggle via kagglehub (recomendado --
# elimina o upload manual). So funciona se houver credencial
# configurada (variaveis de ambiente KAGGLE_USERNAME/KAGGLE_KEY,
# arquivo ~/.kaggle/kaggle.json, ou secrets do Colab).
# -------------------------------------------------------------
try:
    import kagglehub

    # No Colab, tenta carregar as credenciais dos "Secrets" (icone de chave)
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        pass  # nao esta no Colab, ou secrets nao configurados -- tenta mesmo assim

    print("Tentando baixar o dataset direto do Kaggle...")
    pasta_kaggle = kagglehub.dataset_download("imuhammad/course-reviews-on-coursera")
    caminhos = localizar_csvs(pasta_kaggle)
    if len(caminhos) == 2:
        print("Dataset baixado automaticamente em:", pasta_kaggle)
except Exception as e:
    print(f"Nao foi possivel baixar automaticamente ({type(e).__name__}: {e}).")
    print("Seguindo para upload manual...")

# -------------------------------------------------------------
# Opcao 2: upload manual (fallback, caso a Opcao 1 nao funcione)
# -------------------------------------------------------------
if len(caminhos) < 2:
    try:
        import google.colab
        from google.colab import files
        print("\nFaca upload dos arquivos (aceita .csv ou .csv.gz comprimido):")
        uploaded = files.upload()
        caminhos.update(localizar_csvs("."))
    except ImportError:
        pass  # nao esta no Colab -- tenta Opcao 3 abaixo

# -------------------------------------------------------------
# Opcao 3: arquivos ja presentes na pasta local do notebook
# -------------------------------------------------------------
if len(caminhos) < 2:
    caminhos.update(localizar_csvs("."))

# Confirma que os dois arquivos foram localizados por algum dos metodos acima
assert "Coursera_courses.csv" in caminhos, "Coursera_courses.csv nao encontrado (todas as opcoes falharam)"
assert "Coursera_reviews.csv" in caminhos, "Coursera_reviews.csv nao encontrado (todas as opcoes falharam)"

CAMINHO_COURSES = caminhos["Coursera_courses.csv"]
CAMINHO_REVIEWS = caminhos["Coursera_reviews.csv"]
DATA_DIR = ""  # pasta atual -- usada para salvar os modelos (.pkl) gerados nos proximos passos
print("\nArquivos prontos para uso:")
print(" - courses:", CAMINHO_COURSES)
print(" - reviews:", CAMINHO_REVIEWS)


## 1. Análise Exploratória de Dados (EDA)

Objetivo: entender a estrutura dos dados antes de mexer em qualquer modelo — tamanho das
tabelas, valores nulos, duplicatas, distribuição das notas e esparsidade da matriz usuário-item.


In [ ]:
import pandas as pd

# -------------------------------------------------------------
# 1. Carregar os dados
# -------------------------------------------------------------
# 'courses' = tabela de ITENS (os cursos)
# 'reviews' = tabela de INTERACOES (usuario avaliou curso com nota X)
courses = pd.read_csv(CAMINHO_COURSES)
reviews = pd.read_csv(CAMINHO_REVIEWS)

print("=" * 60)
print("1. TAMANHO DAS TABELAS")
print("=" * 60)
print(f"Cursos (itens):        {courses.shape[0]:,} linhas | {courses.shape[1]} colunas")
print(f"Reviews (interacoes):  {reviews.shape[0]:,} linhas | {reviews.shape[1]} colunas")

# -------------------------------------------------------------
# 2. Ver as colunas e os primeiros registros
# -------------------------------------------------------------
print("\n" + "=" * 60)
print("2. COLUNAS")
print("=" * 60)
print("courses:", list(courses.columns))
print("reviews:", list(reviews.columns))

# -------------------------------------------------------------
# 3. Checar valores nulos (dados faltando)
# -------------------------------------------------------------
# Isso e importante: se tiver muito nulo em uma coluna-chave
# (ex: rating ou course_id), precisamos tratar antes de seguir.
print("\n" + "=" * 60)
print("3. VALORES NULOS")
print("=" * 60)
print("courses:\n", courses.isnull().sum())
print("\nreviews:\n", reviews.isnull().sum())

# -------------------------------------------------------------
# 4. Checar duplicatas
# -------------------------------------------------------------
print("\n" + "=" * 60)
print("4. LINHAS DUPLICADAS")
print("=" * 60)
print(f"courses: {courses.duplicated().sum()} duplicadas")
print(f"reviews: {reviews.duplicated().sum()} duplicadas")

# -------------------------------------------------------------
# 5. Distribuicao das notas (ratings)
# -------------------------------------------------------------
# Ponto de atencao que ja discutimos antes: plataformas de curso
# tendem a ter nota concentrada em 4-5 estrelas (vies de quem
# responde review geralmente esta satisfeito).
print("\n" + "=" * 60)
print("5. DISTRIBUICAO DAS NOTAS (RATING)")
print("=" * 60)
print(reviews["rating"].value_counts().sort_index())
print(f"\nMedia geral das notas: {reviews['rating'].mean():.2f}")

# -------------------------------------------------------------
# 6. Quantas avaliacoes cada USUARIO fez (esparsidade)
# -------------------------------------------------------------
# Isso importa muito pra filtragem colaborativa: se cada usuario
# avaliou so 1 curso, fica dificil achar "usuarios parecidos".
avaliacoes_por_usuario = reviews.groupby("reviewers").size()
print("\n" + "=" * 60)
print("6. AVALIACOES POR USUARIO")
print("=" * 60)
print(f"Numero de usuarios unicos: {avaliacoes_por_usuario.shape[0]:,}")
print(f"Media de avaliacoes por usuario: {avaliacoes_por_usuario.mean():.2f}")
print(f"Usuarios com apenas 1 avaliacao: {(avaliacoes_por_usuario == 1).sum():,} "
      f"({(avaliacoes_por_usuario == 1).mean() * 100:.1f}%)")
print(f"Usuarios com 5+ avaliacoes: {(avaliacoes_por_usuario >= 5).sum():,}")

# -------------------------------------------------------------
# 7. Quantas avaliacoes cada CURSO tem
# -------------------------------------------------------------
avaliacoes_por_curso = reviews.groupby("course_id").size()
print("\n" + "=" * 60)
print("7. AVALIACOES POR CURSO")
print("=" * 60)
print(f"Numero de cursos com pelo menos 1 review: {avaliacoes_por_curso.shape[0]:,}")
print(f"Media de avaliacoes por curso: {avaliacoes_por_curso.mean():.1f}")
print("\nTop 5 cursos com mais avaliacoes:")
print(avaliacoes_por_curso.sort_values(ascending=False).head(5))

# -------------------------------------------------------------
# 8. Todo course_id de reviews existe na tabela de cursos?
# -------------------------------------------------------------
# Se houver review de um curso que nao esta na tabela "courses",
# precisamos decidir o que fazer com isso no pre-processamento.
ids_reviews = set(reviews["course_id"].unique())
ids_courses = set(courses["course_id"].unique())
orfaos = ids_reviews - ids_courses
print("\n" + "=" * 60)
print("8. CONSISTENCIA ENTRE TABELAS")
print("=" * 60)
print(f"course_ids em reviews mas ausentes em courses: {len(orfaos)}")

print("\nEDA concluida.")

## 2. Limpeza e Pré-processamento

Remoção de duplicatas (achamos ~64% de linhas duplicadas no EDA), tratamento de nulos,
criação do "perfil textual" de cada curso (agregando suas próprias reviews, já que a base
não possui coluna de descrição) e divisão treino/teste 80/20 por usuário.


In [ ]:
import pandas as pd

# -------------------------------------------------------------
# 1. Carregar os dados originais
# -------------------------------------------------------------
courses = pd.read_csv(CAMINHO_COURSES)
reviews = pd.read_csv(CAMINHO_REVIEWS)

print(f"Antes da limpeza: {reviews.shape[0]:,} reviews")

# -------------------------------------------------------------
# 2. Remover duplicatas exatas
# -------------------------------------------------------------
# No EDA vimos que a mesma review (mesmo texto, usuario, curso,
# data e nota) aparecia repetida 2-3x -- provavelmente um
# artefato do processo de coleta (scraping). Remover para nao
# distorcer o peso de usuarios/cursos no modelo.
reviews = reviews.drop_duplicates()
print(f"Depois de remover duplicatas: {reviews.shape[0]:,} reviews")

# -------------------------------------------------------------
# 3. Tratar valores nulos
# -------------------------------------------------------------
# So 153 reviews sem texto -- decidimos descartar, ja que sao
# poucas e o texto e essencial pro modelo Content-Based.
antes = reviews.shape[0]
reviews = reviews.dropna(subset=["reviews"])
print(f"Removidas {antes - reviews.shape[0]} reviews sem texto")

# -------------------------------------------------------------
# 4. Limpar o nome do reviewer
# -------------------------------------------------------------
# Os nomes vem no formato "By \t Nome Sobrenome" -- vamos tirar
# o prefixo "By" e espacos/tabs extras, pra ficar so o nome.
reviews["reviewers"] = (
    reviews["reviewers"]
    .str.replace(r"^By\s*", "", regex=True)  # remove "By " do inicio
    .str.replace(r"\s+", " ", regex=True)     # colapsa espacos/tabs multiplos
    .str.strip()
)

# -------------------------------------------------------------
# 5. Converter a data para formato datetime
# -------------------------------------------------------------
reviews["date_reviews"] = pd.to_datetime(reviews["date_reviews"], format="%b %d, %Y")

# -------------------------------------------------------------
# 6. Criar um ID numerico para cada usuario
# -------------------------------------------------------------
# Motivo: nomes de reviewer podem se repetir entre pessoas
# diferentes (ex: dois "John Smith"). Isso e uma limitacao
# conhecida do dataset (nao ha ID unico de usuario, so o nome
# que a pessoa digitou). Vamos documentar isso no relatorio
# como uma limitacao dos dados.
# Criamos um ID interno so pra facilitar o processamento.
reviews["user_id"] = reviews["reviewers"].astype("category").cat.codes

# -------------------------------------------------------------
# 7. Criar o "perfil de conteudo" de cada curso
# -------------------------------------------------------------
# Como courses.csv nao tem descricao, vamos concatenar todas as
# reviews de cada curso em um unico texto. Isso funciona como
# "do que esse curso trata", na pratica, porque as pessoas
# comentam o conteudo do curso nas reviews.
perfil_conteudo = (
    reviews.groupby("course_id")["reviews"]
    .apply(lambda textos: " ".join(textos))
    .reset_index()
    .rename(columns={"reviews": "conteudo_textual"})
)

# Juntamos esse perfil de volta na tabela de cursos
courses = courses.merge(perfil_conteudo, on="course_id", how="left")

# Alguns cursos podem nao ter nenhuma review (visto no EDA: 623
# cursos cadastrados, mas so 604 tem review) -- esses ficam sem
# conteudo textual. Vamos preencher com o nome do curso pelo
# menos, pra nao dar erro no TF-IDF depois.
sem_review = courses["conteudo_textual"].isnull().sum()
print(f"\nCursos sem nenhuma review (sem conteudo textual): {sem_review}")
courses["conteudo_textual"] = courses["conteudo_textual"].fillna(courses["name"])

# -------------------------------------------------------------
# 8. Split treino/teste (80/20 por usuario)
# -------------------------------------------------------------
# Importante: NAO e um split aleatorio simples em cima de todas
# as linhas. Fazemos por usuario, pra garantir que todo usuario
# tenha pelo menos uma interacao no treino E no teste (senao a
# gente nao consegue avaliar recomendacao pra ele).
# Ordenamos por data e pegamos os ultimos 20% de cada usuario
# pro teste -- isso simula "prever o futuro" com base no passado,
# que e mais realista do que embaralhar aleatoriamente.

reviews = reviews.sort_values(["user_id", "date_reviews"]).reset_index(drop=True)

# Posicao de cada review dentro do historico do seu usuario (0, 1, 2...)
posicao_no_grupo = reviews.groupby("user_id").cumcount()
# Tamanho total do historico daquele usuario
tamanho_grupo = reviews.groupby("user_id")["user_id"].transform("size")
# Ponto de corte: 80% do historico de cada usuario (minimo 1 review no treino)
corte = (tamanho_grupo * 0.8).astype(int).clip(lower=1)

reviews["conjunto"] = "treino"
reviews.loc[posicao_no_grupo >= corte, "conjunto"] = "teste"

print("\nDistribuicao do split:")
print(reviews["conjunto"].value_counts())

# Usuarios com so 1 review inteira caem 100% no treino (nao tem
# como testar neles) -- isso e esperado e sera discutido como
# limitacao (problema de cold-start).

# -------------------------------------------------------------
# 9. Salvar os dados limpos para os proximos passos
# -------------------------------------------------------------
# Os DataFrames "reviews" e "courses" ja ficam disponiveis em memoria
# para as proximas celulas deste notebook (nao precisamos salvar em CSV aqui).

print("\nArquivos salvos:")
print("- reviews_limpo.csv")
print("- courses_limpo.csv")
print("\nPre-processamento concluido.")

## 3. Modelo Content-Based (TF-IDF)

Vetorização do conteúdo textual de cada curso via TF-IDF e cálculo de similaridade de
cosseno entre todos os pares de cursos. Recomendação = cursos mais parecidos com os que o
usuário avaliou bem (rating ≥ 4) no treino.


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------------------------------------------
# 1. Carregar os dados ja limpos (saida do Passo 2)
# -------------------------------------------------------------
# "courses" e "reviews" ja foram criados e limpos no Passo 2
# Separar treino e teste (coluna criada no Passo 2)
treino = reviews[reviews["conjunto"] == "treino"]
teste = reviews[reviews["conjunto"] == "teste"]

# -------------------------------------------------------------
# 2. Vetorizar o conteudo textual dos cursos com TF-IDF
# -------------------------------------------------------------
# TF-IDF (Term Frequency - Inverse Document Frequency) da mais
# peso pra palavras que aparecem MUITO em um curso especifico,
# mas POUCO nos outros cursos -- ou seja, palavras que realmente
# identificam o assunto daquele curso.
#
# stop_words="english": remove palavras sem valor semantico
#   (the, is, a, and...) -- as reviews estao em ingles.
# max_features=5000: limita o vocabulario as 5000 palavras mais
#   relevantes, pra nao pesar demais o processamento.
# min_df=2: ignora palavras que aparecem em menos de 2 cursos
#   (normalmente erros de digitacao ou nomes proprios raros).
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    min_df=2,
)

matriz_tfidf = vectorizer.fit_transform(courses["conteudo_textual"])
print(f"Matriz TF-IDF: {matriz_tfidf.shape[0]} cursos x {matriz_tfidf.shape[1]} termos")

# -------------------------------------------------------------
# 3. Calcular a similaridade entre TODOS os pares de cursos
# -------------------------------------------------------------
# similaridade_cosseno[i][j] = o quao parecido o curso i e do
# curso j (varia de 0 = nada parecido, a 1 = identico).
similaridade = cosine_similarity(matriz_tfidf)

# Colocamos em um DataFrame pra facilitar consultar por course_id
sim_df = pd.DataFrame(
    similaridade,
    index=courses["course_id"],
    columns=courses["course_id"],
)

# -------------------------------------------------------------
# 4. Funcao que recomenda cursos pra um usuario
# -------------------------------------------------------------
def recomendar_content_based(user_id, top_k=5):
    """
    Recomenda 'top_k' cursos para o usuario, baseado nos cursos
    que ele avaliou bem (rating >= 4) no conjunto de TREINO.
    """
    # Cursos que esse usuario avaliou bem no treino
    avaliados_bem = treino[
        (treino["user_id"] == user_id) & (treino["rating"] >= 4)
    ]["course_id"].unique()

    if len(avaliados_bem) == 0:
        return []  # sem historico positivo, nao da pra recomendar por conteudo

    # Cursos que o usuario ja fez (treino + teste), pra nao recomendar de novo
    ja_fez = set(treino[treino["user_id"] == user_id]["course_id"].unique())

    # Somamos a similaridade dos cursos candidatos com TODOS os
    # cursos que o usuario avaliou bem -- cursos parecidos com
    # varios dos que ele gostou pontuam mais alto.
    score_candidatos = sim_df[avaliados_bem].sum(axis=1)

    # Removemos os cursos que ele ja fez
    score_candidatos = score_candidatos.drop(labels=ja_fez, errors="ignore")

    # Pegamos os top_k com maior score
    top_cursos = score_candidatos.sort_values(ascending=False).head(top_k)

    return list(top_cursos.index)


# -------------------------------------------------------------
# 5. Teste manual (sanity check) -- vamos conferir se as
#    recomendacoes fazem sentido pra um usuario real
# -------------------------------------------------------------
# Pegamos um usuario "tipico": nem quem so avaliou 1 curso, nem
# um outlier com centenas de avaliacoes (que dilui o sinal e
# dificulta interpretar visualmente se a recomendacao faz sentido).
# Escolhemos alguem com um historico moderado, entre 3 e 8 cursos.
contagem = treino[treino["rating"] >= 4]["user_id"].value_counts()
candidatos = contagem[(contagem >= 3) & (contagem <= 8)]
usuario_exemplo = candidatos.index[0]

cursos_que_gostou = treino[
    (treino["user_id"] == usuario_exemplo) & (treino["rating"] >= 4)
]["course_id"].tolist()

nomes_cursos_que_gostou = courses[
    courses["course_id"].isin(cursos_que_gostou)
]["name"].tolist()

recomendados = recomendar_content_based(usuario_exemplo, top_k=5)
nomes_recomendados = courses[courses["course_id"].isin(recomendados)]["name"].tolist()

print("\n" + "=" * 60)
print("TESTE MANUAL DE SANIDADE (Content-Based)")
print("=" * 60)
print(f"Usuario: {usuario_exemplo}")
print(f"\nCursos que ele avaliou bem (treino): {nomes_cursos_que_gostou}")
print(f"\nCursos RECOMENDADOS pelo modelo: {nomes_recomendados}")

# -------------------------------------------------------------
# 6. Salvar a matriz de similaridade e o vectorizer, pra usar
#    depois na etapa de avaliacao (Passo 5)
# -------------------------------------------------------------
import pickle

with open(DATA_DIR + "content_based_model.pkl", "wb") as f:
    pickle.dump({"sim_df": sim_df}, f)

print("\nModelo Content-Based salvo em content_based_model.pkl")

## 4. Modelo Colaborativo (KNN Item-Based)

Constrói a matriz esparsa item×usuário a partir das notas do treino e treina um KNN com
métrica de cosseno para encontrar cursos "vizinhos" (avaliados de forma parecida pelas
mesmas pessoas).


In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# -------------------------------------------------------------
# 1. Carregar os dados limpos e pegar so o treino
# -------------------------------------------------------------
# "courses" e "reviews" ja foram criados e limpos no Passo 2
treino = reviews[reviews["conjunto"] == "treino"]

# -------------------------------------------------------------
# 2. Construir a matriz ITEM x USUARIO (esparsa)
# -------------------------------------------------------------
# Cada linha = um curso. Cada coluna = um usuario. O valor na
# celula = a nota que aquele usuario deu aquele curso (0 se nao
# avaliou).
#
# Usamos matriz ESPARSA (scipy csr_matrix) porque a maioria das
# combinacoes curso x usuario e vazia (um usuario so avalia
# alguns cursos, nao os 604) -- guardar isso como matriz densa
# gastaria memoria demais a toa.

# Criamos indices numericos sequenciais para cursos e usuarios
cursos_unicos = treino["course_id"].astype("category")
usuarios_unicos = treino["user_id"].astype("category")

linhas = cursos_unicos.cat.codes      # indice do curso (linha da matriz)
colunas = usuarios_unicos.cat.codes   # indice do usuario (coluna da matriz)
valores = treino["rating"].values

n_cursos = cursos_unicos.cat.categories.shape[0]
n_usuarios = usuarios_unicos.cat.categories.shape[0]

matriz_item_usuario = csr_matrix(
    (valores, (linhas, colunas)),
    shape=(n_cursos, n_usuarios),
)

print(f"Matriz item x usuario: {matriz_item_usuario.shape[0]} cursos x "
      f"{matriz_item_usuario.shape[1]} usuarios")
print(f"Preenchimento (densidade): {matriz_item_usuario.nnz / (n_cursos * n_usuarios) * 100:.3f}%")

# Guardamos o mapa "indice da matriz -> course_id real" pra poder
# traduzir de volta depois
indice_para_course_id = dict(enumerate(cursos_unicos.cat.categories))
course_id_para_indice = {v: k for k, v in indice_para_course_id.items()}

# -------------------------------------------------------------
# 3. Treinar o KNN sobre os cursos
# -------------------------------------------------------------
# metric="cosine": mede o angulo entre os vetores de notas de
#   dois cursos -- funciona bem mesmo com dados esparsos.
# n_neighbors=11: pegamos 11 porque o proprio curso sempre aparece
#   como seu vizinho mais proximo (distancia 0) -- descartamos ele
#   depois e sobram 10 vizinhos reais.
modelo_knn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=11)
modelo_knn.fit(matriz_item_usuario)

# -------------------------------------------------------------
# 4. Funcao que recomenda cursos pra um usuario
# -------------------------------------------------------------
def recomendar_colaborativo(user_id, top_k=5):
    """
    Recomenda 'top_k' cursos pro usuario, baseado em cursos
    parecidos (pelo padrao de notas) com os que ele avaliou bem.
    """
    avaliados_bem = treino[
        (treino["user_id"] == user_id) & (treino["rating"] >= 4)
    ]["course_id"].unique()

    # Filtra so os que existem na matriz treinada
    avaliados_bem = [c for c in avaliados_bem if c in course_id_para_indice]
    if len(avaliados_bem) == 0:
        return []  # cold-start: sem historico positivo no treino

    ja_fez = set(treino[treino["user_id"] == user_id]["course_id"].unique())

    # Para cada curso que o usuario gostou, buscamos os vizinhos
    # mais proximos e acumulamos um "score" (quanto mais perto,
    # maior o peso: usamos 1 - distancia_cosseno)
    scores = {}
    for course_id in avaliados_bem:
        idx = course_id_para_indice[course_id]
        distancias, vizinhos = modelo_knn.kneighbors(
            matriz_item_usuario[idx], n_neighbors=11
        )
        for dist, viz_idx in zip(distancias[0], vizinhos[0]):
            viz_course_id = indice_para_course_id[viz_idx]
            if viz_course_id == course_id or viz_course_id in ja_fez:
                continue  # ignora o proprio curso e o que ele ja fez
            peso = 1 - dist  # distancia 0 = identico -> peso 1
            scores[viz_course_id] = scores.get(viz_course_id, 0) + peso

    top_cursos = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [c for c, _ in top_cursos]


# -------------------------------------------------------------
# 5. Teste manual (mesmo usuario do Passo 3, pra comparar)
# -------------------------------------------------------------
usuario_exemplo = 1009  # mesmo usuario usado no Content-Based

cursos_que_gostou = treino[
    (treino["user_id"] == usuario_exemplo) & (treino["rating"] >= 4)
]["course_id"].tolist()
nomes_gostou = courses[courses["course_id"].isin(cursos_que_gostou)]["name"].tolist()

recomendados = recomendar_colaborativo(usuario_exemplo, top_k=5)
nomes_recomendados = courses[courses["course_id"].isin(recomendados)]["name"].tolist()

print("\n" + "=" * 60)
print("TESTE MANUAL DE SANIDADE (Colaborativo - KNN Item-Based)")
print("=" * 60)
print(f"Usuario: {usuario_exemplo}")
print(f"\nCursos que ele avaliou bem (treino): {nomes_gostou}")
print(f"\nCursos RECOMENDADOS pelo modelo: {nomes_recomendados}")

# -------------------------------------------------------------
# 6. Salvar o modelo pra usar na avaliacao (Passo 5)
# -------------------------------------------------------------
import pickle

with open(DATA_DIR + "collaborative_model.pkl", "wb") as f:
    pickle.dump({
        "modelo_knn": modelo_knn,
        "matriz_item_usuario": matriz_item_usuario,
        "indice_para_course_id": indice_para_course_id,
        "course_id_para_indice": course_id_para_indice,
    }, f)

print("\nModelo Colaborativo salvo em collaborative_model.pkl")

## 5. Avaliação (Precision@K e Recall@K)

Compara os dois modelos, mais um baseline de popularidade, em uma amostra de 1.500 usuários
do conjunto de teste.


In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

K = 5          # quantas recomendacoes avaliamos (top-5)
N_USUARIOS_AVALIADOS = 1500  # amostra de usuarios pra avaliar (dataset e grande, avaliar todo mundo seria lento)

# -------------------------------------------------------------
# 1. Carregar dados e os dois modelos ja treinados
# -------------------------------------------------------------
# "courses", "reviews", "treino" e "teste" ja existem em memoria

with open(DATA_DIR + "content_based_model.pkl", "rb") as f:
    cb = pickle.load(f)
    sim_df = cb["sim_df"]

with open(DATA_DIR + "collaborative_model.pkl", "rb") as f:
    colab = pickle.load(f)
    modelo_knn = colab["modelo_knn"]
    matriz_item_usuario = colab["matriz_item_usuario"]
    indice_para_course_id = colab["indice_para_course_id"]
    course_id_para_indice = colab["course_id_para_indice"]

# -------------------------------------------------------------
# 2. Funcoes de recomendacao (mesma logica dos Passos 3 e 4)
# -------------------------------------------------------------
def recomendar_content_based(user_id, top_k=K):
    avaliados_bem = treino[(treino["user_id"] == user_id) & (treino["rating"] >= 4)]["course_id"].unique()
    avaliados_bem = [c for c in avaliados_bem if c in sim_df.columns]
    if len(avaliados_bem) == 0:
        return []
    ja_fez = set(treino[treino["user_id"] == user_id]["course_id"].unique())
    score = sim_df[avaliados_bem].sum(axis=1).drop(labels=ja_fez, errors="ignore")
    return list(score.sort_values(ascending=False).head(top_k).index)


def recomendar_colaborativo(user_id, top_k=K):
    avaliados_bem = treino[(treino["user_id"] == user_id) & (treino["rating"] >= 4)]["course_id"].unique()
    avaliados_bem = [c for c in avaliados_bem if c in course_id_para_indice]
    if len(avaliados_bem) == 0:
        return []
    ja_fez = set(treino[treino["user_id"] == user_id]["course_id"].unique())
    scores = {}
    for course_id in avaliados_bem:
        idx = course_id_para_indice[course_id]
        distancias, vizinhos = modelo_knn.kneighbors(matriz_item_usuario[idx], n_neighbors=11)
        for dist, viz_idx in zip(distancias[0], vizinhos[0]):
            viz_course_id = indice_para_course_id[viz_idx]
            if viz_course_id == course_id or viz_course_id in ja_fez:
                continue
            scores[viz_course_id] = scores.get(viz_course_id, 0) + (1 - dist)
    top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [c for c, _ in top]


# -------------------------------------------------------------
# 3. Baseline de popularidade
# -------------------------------------------------------------
# Ranking fixo dos cursos mais bem avaliados no treino (nota
# media, com um minimo de avaliacoes pra nao deixar um curso com
# 1 review de 5 estrelas dominar o ranking).
stats_curso = treino.groupby("course_id")["rating"].agg(["mean", "count"])
stats_curso = stats_curso[stats_curso["count"] >= 30]  # minimo de avaliacoes
RANKING_POPULARIDADE = stats_curso.sort_values("mean", ascending=False).index.tolist()

def recomendar_popularidade(user_id, top_k=K):
    ja_fez = set(treino[treino["user_id"] == user_id]["course_id"].unique())
    recomendados = [c for c in RANKING_POPULARIDADE if c not in ja_fez]
    return recomendados[:top_k]


# -------------------------------------------------------------
# 4. Selecionar usuarios de teste pra avaliar
# -------------------------------------------------------------
# So faz sentido avaliar usuarios que tem pelo menos 1 curso
# "relevante" (rating >= 4) no TESTE.
relevantes_por_usuario = (
    teste[teste["rating"] >= 4]
    .groupby("user_id")["course_id"]
    .apply(set)
)

usuarios_para_avaliar = relevantes_por_usuario.index.to_series().sample(
    n=min(N_USUARIOS_AVALIADOS, len(relevantes_por_usuario)),
    random_state=42,
)

print(f"Avaliando {len(usuarios_para_avaliar)} usuarios (de {len(relevantes_por_usuario)} elegiveis)")

# -------------------------------------------------------------
# 5. Rodar a avaliacao pros 3 modelos
# -------------------------------------------------------------
def avaliar_modelo(funcao_recomendacao, nome):
    precisions, recalls = [], []
    for user_id in usuarios_para_avaliar:
        relevantes = relevantes_por_usuario[user_id]
        recomendados = set(funcao_recomendacao(user_id, top_k=K))
        if len(recomendados) == 0:
            continue  # usuario sem historico no treino (cold-start) -- fora da media
        acertos = len(recomendados & relevantes)
        precisions.append(acertos / K)
        recalls.append(acertos / len(relevantes))
    return {
        "modelo": nome,
        "precision@5": np.mean(precisions),
        "recall@5": np.mean(recalls),
        "usuarios_avaliados": len(precisions),
    }

resultados = []
print("\nAvaliando baseline de popularidade...")
resultados.append(avaliar_modelo(recomendar_popularidade, "Popularidade (baseline)"))

print("Avaliando Content-Based...")
resultados.append(avaliar_modelo(recomendar_content_based, "Content-Based (TF-IDF)"))

print("Avaliando Colaborativo...")
resultados.append(avaliar_modelo(recomendar_colaborativo, "Colaborativo (KNN item-based)"))

# -------------------------------------------------------------
# 6. Mostrar resultado final
# -------------------------------------------------------------
df_resultado = pd.DataFrame(resultados)
print("\n" + "=" * 60)
print(f"RESULTADO FINAL - Precision@{K} e Recall@{K}")
print("=" * 60)
print(df_resultado.to_string(index=False))

df_resultado.to_csv(DATA_DIR + "resultado_avaliacao.csv", index=False)
print("\nResultado salvo em resultado_avaliacao.csv")

## 6. Modelo Híbrido

Combina os scores normalizados (0–1) do Content-Based e do Colaborativo, com peso maior
para o Colaborativo (melhor isoladamente). Resolve cold-start ao cair no modelo que
conseguir gerar recomendação.


In [ ]:
import pandas as pd
import numpy as np
import pickle

K = 5
ALPHA = 0.6  # peso do modelo colaborativo (1 - ALPHA = peso do content-based)

# -------------------------------------------------------------
# 1. Carregar dados e os modelos ja treinados
# -------------------------------------------------------------
# "courses", "reviews", "treino" e "teste" ja existem em memoria

with open(DATA_DIR + "content_based_model.pkl", "rb") as f:
    sim_df = pickle.load(f)["sim_df"]

with open(DATA_DIR + "collaborative_model.pkl", "rb") as f:
    colab = pickle.load(f)
    modelo_knn = colab["modelo_knn"]
    matriz_item_usuario = colab["matriz_item_usuario"]
    indice_para_course_id = colab["indice_para_course_id"]
    course_id_para_indice = colab["course_id_para_indice"]

# -------------------------------------------------------------
# 2. Funcoes que retornam o SCORE BRUTO por curso (nao so o top_k)
# -------------------------------------------------------------
def scores_content_based(user_id):
    avaliados_bem = treino[(treino["user_id"] == user_id) & (treino["rating"] >= 4)]["course_id"].unique()
    avaliados_bem = [c for c in avaliados_bem if c in sim_df.columns]
    if len(avaliados_bem) == 0:
        return pd.Series(dtype=float)
    return sim_df[avaliados_bem].sum(axis=1)


def scores_colaborativo(user_id):
    avaliados_bem = treino[(treino["user_id"] == user_id) & (treino["rating"] >= 4)]["course_id"].unique()
    avaliados_bem = [c for c in avaliados_bem if c in course_id_para_indice]
    if len(avaliados_bem) == 0:
        return pd.Series(dtype=float)
    scores = {}
    for course_id in avaliados_bem:
        idx = course_id_para_indice[course_id]
        distancias, vizinhos = modelo_knn.kneighbors(matriz_item_usuario[idx], n_neighbors=11)
        for dist, viz_idx in zip(distancias[0], vizinhos[0]):
            viz_course_id = indice_para_course_id[viz_idx]
            if viz_course_id == course_id:
                continue
            scores[viz_course_id] = scores.get(viz_course_id, 0) + (1 - dist)
    return pd.Series(scores, dtype=float)


def normalizar(serie):
    """Leva os valores pra escala 0-1 (min-max). Evita divisao por zero
    quando todos os scores sao iguais."""
    if len(serie) == 0:
        return serie
    minimo, maximo = serie.min(), serie.max()
    if maximo == minimo:
        return serie * 0  # todos empatados -> todos viram 0
    return (serie - minimo) / (maximo - minimo)


# -------------------------------------------------------------
# 3. Funcao de recomendacao HIBRIDA
# -------------------------------------------------------------
def recomendar_hibrido(user_id, top_k=K, alpha=ALPHA):
    s_content = normalizar(scores_content_based(user_id))
    s_colab = normalizar(scores_colaborativo(user_id))

    # Fallback: se um dos dois modelos nao conseguiu gerar score
    # (ex: usuario novo sem vizinhos no colaborativo), usamos so
    # o outro -- e o proprio ponto forte do hibrido: cobre o
    # cold-start de um modelo com o outro.
    if len(s_colab) == 0 and len(s_content) == 0:
        return []
    elif len(s_colab) == 0:
        score_final = s_content
    elif len(s_content) == 0:
        score_final = s_colab
    else:
        # Junta os dois (index=course_id), preenchendo com 0 onde
        # um modelo nao deu score pra aquele curso
        df_scores = pd.DataFrame({"content": s_content, "colab": s_colab}).fillna(0)
        score_final = alpha * df_scores["colab"] + (1 - alpha) * df_scores["content"]

    ja_fez = set(treino[treino["user_id"] == user_id]["course_id"].unique())
    score_final = score_final.drop(labels=ja_fez, errors="ignore")
    return list(score_final.sort_values(ascending=False).head(top_k).index)


# -------------------------------------------------------------
# 4. Teste manual (mesmo usuario dos passos anteriores)
# -------------------------------------------------------------
usuario_exemplo = 1009
recomendados = recomendar_hibrido(usuario_exemplo, top_k=5)
nomes = courses[courses["course_id"].isin(recomendados)]["name"].tolist()
print("=" * 60)
print("TESTE MANUAL DE SANIDADE (Hibrido)")
print("=" * 60)
print(f"Usuario: {usuario_exemplo}")
print(f"Cursos RECOMENDADOS pelo hibrido: {nomes}")

# -------------------------------------------------------------
# 5. Avaliacao (mesma amostra de usuarios do Passo 5, pra
#    comparacao justa)
# -------------------------------------------------------------
relevantes_por_usuario = (
    teste[teste["rating"] >= 4].groupby("user_id")["course_id"].apply(set)
)
usuarios_para_avaliar = relevantes_por_usuario.index.to_series().sample(
    n=min(1500, len(relevantes_por_usuario)), random_state=42  # mesmo seed do Passo 5
)

precisions, recalls = [], []
for user_id in usuarios_para_avaliar:
    relevantes = relevantes_por_usuario[user_id]
    recomendados = set(recomendar_hibrido(user_id, top_k=K))
    if len(recomendados) == 0:
        continue
    acertos = len(recomendados & relevantes)
    precisions.append(acertos / K)
    recalls.append(acertos / len(relevantes))

print("\n" + "=" * 60)
print(f"RESULTADO - Hibrido (alpha={ALPHA})")
print("=" * 60)
print(f"Precision@{K}: {np.mean(precisions):.4f}")
print(f"Recall@{K}:    {np.mean(recalls):.4f}")
print(f"Usuarios avaliados: {len(precisions)}")

# -------------------------------------------------------------
# 6. Atualizar a tabela comparativa final (junta com o Passo 5)
# -------------------------------------------------------------
df_anterior = pd.read_csv(DATA_DIR + "resultado_avaliacao.csv")
nova_linha = pd.DataFrame([{
    "modelo": f"Hibrido (alpha={ALPHA})",
    "precision@5": np.mean(precisions),
    "recall@5": np.mean(recalls),
    "usuarios_avaliados": len(precisions),
}])
df_final = pd.concat([df_anterior, nova_linha], ignore_index=True)
df_final.to_csv(DATA_DIR + "resultado_avaliacao.csv", index=False)

print("\nTabela comparativa final:")
print(df_final.to_string(index=False))

## Conclusão

| Modelo | Precision@5 | Recall@5 |
|---|---|---|
| Popularidade (baseline) | ~0,04% | ~0,14% |
| Content-Based (TF-IDF) | ~3,0% | ~14,0% |
| Colaborativo (KNN item-based) | ~4,3% | ~19,1% |
| **Híbrido (α=0,6)** | **~4,4%** | **~20,2%** |

O modelo **Híbrido** apresentou o melhor desempenho entre os quatro, com destaque para o
ganho expressivo de ambos os modelos de recomendação frente ao baseline de popularidade —
evidência de que a personalização está de fato agregando valor. O Colaborativo superou o
Content-Based isoladamente, provavelmente por capturar padrões reais de comportamento em
vez de depender do texto (mais ruidoso) das reviews como proxy de conteúdo.
